# PhytoVeda — Full Pipeline on Google Colab

**Multimodal Pharmacognosy: AI-Driven Botanical Identification and Textual Mapping in Ayurveda**

This notebook runs the complete PhytoVeda pipeline on Google Colab:

| Step | Description | Storage |
|------|-------------|--------|
| 1. Environment Setup | Mount Drive, install packages, verify GPU | — |
| 2. Dataset Download | Download 6 datasets to Colab SSD | `/content/datasets/` (SSD) |
| 3. Data Pipeline | Build federated dataset + augmentation | SSD |
| 4. Training | Train HierViT-CMTL with AMP | Checkpoints → Drive |
| 5. Evaluation | F1, accuracy, confusion matrices | Results → Drive |
| 6. Inference | Single-image prediction + Dosha mapping | — |
| 7. RAG Pipeline | Index Ayurvedic texts, generate reports | ChromaDB → Drive, Reports → Drive |
| 8. Active Learning | Uncertainty sampling, oracle pipeline | Quarantine → Drive |
| 9. Enterprise Features | Conservation, traceability, formulation | All → Drive |

**Storage strategy**: Datasets on Colab SSD (`/content/`) for fast I/O. Everything else persists to Google Drive.

---
## 1. Environment Setup

### 1a. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 1b. Clone / Sync Repository

If PhytoVeda is already on your Drive, skip the clone. Otherwise, clone it to Drive for persistence.

In [ ]:
import os
from pathlib import Path

REPO_DIR = Path("/content/drive/MyDrive/PhytoVeda")

if not REPO_DIR.exists():
    # Clone fresh — replace with your repo URL
    !git clone https://github.com/kautilyaa/PhytoVeda.git "{REPO_DIR}"
else:
    print(f"Repository already exists at {REPO_DIR}")
    # Optional: pull latest changes
    # !cd "{REPO_DIR}" && git pull

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

### 1c. Install PhytoVeda + Dependencies

In [ ]:
import subprocess, sys, os

# Install core dependencies
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "torch", "torchvision", "timm", "transformers",
    "numpy", "Pillow", "scikit-learn", "pandas",
    "albumentations", "pyyaml", "pydantic",
    "chromadb", "sentence-transformers",
    "google-generativeai", "google-cloud-storage", "google-cloud-aiplatform",
    "fastapi", "uvicorn[standard]", "python-multipart",
    "wandb",
])

# Add PhytoVeda source to Python path (no editable install needed)
src_path = os.path.join(os.getcwd(), "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Verify
import phytoveda
print("PhytoVeda loaded successfully from:", phytoveda.__file__)


### 1d. Initialize DriveManager + Verify Environment

In [ ]:
from phytoveda.colab import DriveManager, ColabEnvironment

# Initialize Drive-backed storage manager
dm = DriveManager()
dm.scaffold()  # Creates all directories on SSD + Drive

# Environment report
env = ColabEnvironment()
print(env.summary())
print()
print(dm.summary())

In [ ]:
# Verify readiness
checks = env.check_ready()
for check, ok in checks.items():
    status = "OK" if ok else "MISSING"
    print(f"  {check:20s} {status}")

if not checks.get("gpu_available"):
    print("\nWARNING: No GPU detected. Go to Runtime > Change runtime type > GPU.")

In [ ]:
# Configure environment
env.configure_cuda()
env.set_seed(42)

# Uncomment for Python 3.14+ free-threaded builds:
# env.configure_free_threading()

---
## 2. Dataset Download

Downloads all 6 datasets to Colab SSD (`/content/datasets/`) for maximum I/O speed during training. This is ephemeral — datasets need to be re-downloaded on runtime restart.

**Datasets**: Herbify (91 species), Assam (10), AI-MedLeafX (4 + pathology), CIMPD (23), SIMPD V1 (20) — [Mendeley](https://data.mendeley.com/datasets/9d89vjcghv/2), EarlyNSD (3 + deficiency)

In [ ]:
from pathlib import Path
from phytoveda.data.download import download_all, validate_all

DATA_ROOT = Path(dm.datasets_dir)
# Scan Drive folder for pre-downloaded ZIPs (searches subdirectories too)
ZIP_DIRS = ["/content/drive/MyDrive/PhytoVedaData"]

print(f"Downloading datasets to: {DATA_ROOT}")
print(f"Scanning for ZIPs in: {ZIP_DIRS}")

download_all(DATA_ROOT, zip_dirs=ZIP_DIRS)

results = validate_all(DATA_ROOT)
for key, (ok, msg) in results.items():
    print(f"[{'OK' if ok else 'MISSING'}] {key}: {msg}")

In [ ]:
# Verify downloads with image counts
from phytoveda.data.download import validate_all, count_images, DATASET_SOURCES

results = validate_all(DATA_ROOT)
for key, (ok, msg) in results.items():
    ds_dir = DATA_ROOT / key
    n_images = count_images(ds_dir) if ds_dir.exists() else 0
    expected = DATASET_SOURCES[key].expected_images
    status = 'OK' if ok else 'MISSING'
    print(f"  {key:15s} {status:<8s} ({n_images:>6} / {expected:>6} images)")

### 2b. Dataset Cache (Skip Re-Downloads on Restart)

Cache download status to Drive so restarts can skip already-downloaded datasets.

In [ ]:
from phytoveda.colab import DatasetCache

# Initialize cache on Drive
data_cache = DatasetCache(cache_dir=dm.drive_base / "cache")
print(data_cache.summary())
print("\n(Split caching will run after dataloaders are built)")

---
## 3. Data Pipeline

Build the federated dataset from all 6 sources with unified taxonomy, augmentation, and weighted sampling.

In [ ]:
import yaml
from phytoveda.data.datasets import build_dataloaders
from phytoveda.data.taxonomy import SpeciesTaxonomy, PATHOLOGY_CLASSES, map_pathology_label

# Load config (needed by dataloaders, model, and training)
with open("configs/hiervit_cmtl.yaml") as f:
    config = yaml.safe_load(f)

print(f"Backbone:         {config['model']['backbone']}")
print(f"Image size:       {config['data']['image_size']}")
print(f"Batch size:       {config['training']['batch_size']}")
print(f"Pathology classes: {len(PATHOLOGY_CLASSES)}")
print(f"Pathology labels:  {PATHOLOGY_CLASSES}")

In [ ]:
# Build data loaders — data on SSD for fast I/O
train_loader, val_loader, test_loader, species_tax = build_dataloaders(
    data_root=str(DATA_ROOT),
    batch_size=config["training"]["batch_size"],
    num_workers=config["data"]["num_workers"],
    image_size=config["data"]["image_size"],
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")
print(f"Species:       {species_tax.num_species}")

In [ ]:
# Preview a batch
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF
from phytoveda.data.taxonomy import ID_TO_PATHOLOGY

images, species_ids, pathology_ids = next(iter(train_loader))
print(f"Batch shape: {images.shape}")
print(f"Species IDs: {species_ids[:8].tolist()}")
print(f"Pathology IDs: {pathology_ids[:8].tolist()}")

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    if i < len(images):
        img = images[i].clone()
        img = img * 0.226 + 0.456
        img = img.clamp(0, 1)
        ax.imshow(TF.to_pil_image(img))
        sp = species_tax.get_name(species_ids[i].item())
        pa = ID_TO_PATHOLOGY.get(pathology_ids[i].item(), "?")
        ax.set_title(f"{sp}\n{pa}", fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.savefig(str(dm.results_dir / "batch_preview.png"), dpi=150)
plt.show()
print(f"Saved to {dm.results_dir / 'batch_preview.png'}")

---
## 4. Training (Colab-Optimized)

Uses `ColabTrainer` with:
- **Gradient accumulation** — simulate large batches on limited GPU memory
- **Auto batch size finder** — detect max batch that fits your GPU
- **torch.compile** — speed up training on A100/T4
- **Crash-resilient checkpointing** — save every 15 min (not just per-epoch)
- **GPU memory monitoring** — real-time VRAM tracking

Checkpoints saved to Google Drive so they survive runtime restarts.

In [ ]:
from phytoveda.training.trainer import TrainConfig
from phytoveda.colab import ColabTrainer, GPUMonitor, find_max_batch_size, compile_model

# Load config with checkpoint dir pointing to Drive
train_config = TrainConfig.from_yaml("configs/hiervit_cmtl.yaml")
train_config.checkpoint_dir = str(dm.checkpoints_dir)

print(f"Training config:")
print(f"  Epochs:         {train_config.epochs}")
print(f"  Batch size:     {train_config.batch_size}")
print(f"  Learning rate:  {train_config.learning_rate}")
print(f"  Mixed precision: {train_config.mixed_precision}")
print(f"  Checkpoint dir: {train_config.checkpoint_dir}")
print(f"  Device:         {env.device()}")

In [ ]:
# Build model
from phytoveda.models.cmtl import HierViTCMTL
from phytoveda.data.taxonomy import PATHOLOGY_CLASSES

model = HierViTCMTL(
    num_species=species_tax.num_species,
    num_pathologies=len(PATHOLOGY_CLASSES),
    backbone_name=config["model"]["backbone"],
    pretrained=config["model"]["pretrained"],
    species_hidden_dim=config["model"]["species_hidden_dim"],
    pathology_hidden_dim=config["model"]["pathology_hidden_dim"],
    dropout=config["model"]["dropout"],
)

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable:,}")

In [ ]:
# ── Gradient Accumulation ──
# If model OOMs at batch_size=16, use accumulation:
#   batch_size=4, accumulation_steps=4 → effective batch of 16
#   batch_size=2, accumulation_steps=8 → effective batch of 16
ACCUMULATION_STEPS = 1  # Set > 1 if you get OOM

# ── torch.compile (optional, A100 only) ──
# Uncomment on A100 for 10-30% speedup. Skip on T4/L4.
# from phytoveda.colab import compile_model
# model = compile_model(model, mode="reduce-overhead", fallback=True)

print(f"Effective batch size: {train_config.batch_size} x {ACCUMULATION_STEPS} = "
      f"{train_config.batch_size * ACCUMULATION_STEPS}")

In [ ]:
# Resume from Drive checkpoint if available
resume_path = dm.latest_checkpoint()
if resume_path:
    print(f"Resuming from: {resume_path}")
else:
    print("Starting fresh training")

In [ ]:
# ── Train with ColabTrainer ──
# Includes gradient accumulation, crash checkpointing, GPU monitoring

colab_trainer = ColabTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=train_config,
    checkpoint_dir=dm.checkpoints_dir,
    device=env.device(),
    accumulation_steps=ACCUMULATION_STEPS,
    crash_interval_minutes=15,  # Auto-save every 15 min for crash resilience
)

# Auto-resume from crash checkpoint if runtime was disconnected
if colab_trainer.resume_from_crash():
    print("Resumed from crash recovery checkpoint!")
elif resume_path:
    import torch
    ckpt = torch.load(resume_path, map_location=env.device(), weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    colab_trainer.optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    print(f"Resumed from: {resume_path}")

history = colab_trainer.train()

# Cache training history to Drive
data_cache.save_history(history)
print(f"\nCheckpoints saved to: {dm.checkpoints_dir}")

In [ ]:
# Plot training curves and save to Drive
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

# Loss
axes[0].plot(history['train_loss'], label='Total Loss')
axes[0].plot(history['species_loss'], label='Species', alpha=0.7)
axes[0].plot(history['disease_loss'], label='Disease', alpha=0.7)
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

# F1
axes[1].plot(history['species_f1'], label='Species F1')
axes[1].plot(history['pathology_f1'], label='Pathology F1')
axes[1].plot(history['avg_f1'], label='Average F1', linewidth=2)
axes[1].set_title('F1 Score')
axes[1].set_xlabel('Epoch')
axes[1].legend()

# Learning rate
axes[2].plot(history['lr'])
axes[2].set_title('Learning Rate')
axes[2].set_xlabel('Epoch')

# GPU memory
if any(v > 0 for v in history.get('gpu_peak_mb', [])):
    axes[3].plot(history['gpu_peak_mb'])
    axes[3].set_title('GPU Peak Memory (MB)')
    axes[3].set_xlabel('Epoch')
else:
    axes[3].text(0.5, 0.5, 'No GPU', ha='center', va='center', fontsize=14)
    axes[3].set_title('GPU Memory')

plt.tight_layout()
plt.savefig(str(dm.results_dir / "training_curves.png"), dpi=150)
plt.show()
print(f"Saved to {dm.results_dir / 'training_curves.png'}")

---
## 5. Evaluation

Evaluate the trained model on the test set. Results saved to Drive.

In [ ]:
from phytoveda.training.evaluation import evaluate, evaluate_detailed
import torch
import json

# Load best checkpoint from Drive
best_ckpt = dm.latest_checkpoint()
print(f"Evaluating: {best_ckpt}")

device = env.device()
checkpoint = torch.load(best_ckpt, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

# Standard evaluation
metrics = evaluate(model, test_loader, device=device)
print("\nTest Metrics:")
for k, v in metrics.items():
    print(f"  {k:25s} {v:.4f}")

# Save metrics to Drive
metrics_path = dm.results_dir / "test_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"\nMetrics saved to {metrics_path}")

In [ ]:
# Detailed per-class evaluation
from phytoveda.data.taxonomy import PATHOLOGY_CLASSES

species_names = [species_tax.get_name(i) for i in range(species_tax.num_species)]
pathology_names = PATHOLOGY_CLASSES

detailed = evaluate_detailed(
    model, test_loader, device=device,
    species_names=species_names,
    pathology_names=pathology_names,
)

print("\nPathology Per-Class Report:")
print(detailed["pathology_report"])

# Save detailed report to Drive
report_path = dm.results_dir / "detailed_evaluation.json"
with open(report_path, "w") as f:
    json.dump({
        "species_report": detailed["species_report"],
        "pathology_report": detailed["pathology_report"],
    }, f, indent=2)
print(f"Saved to {report_path}")

---
## 6. Single-Image Inference

Load the trained model and run inference on a single leaf image.

In [ ]:
from phytoveda.api.inference import InferencePipeline
from PIL import Image

# Load pipeline from Drive checkpoint
pipeline = InferencePipeline.from_checkpoint(
    checkpoint_path=str(dm.latest_checkpoint()),
    device=env.device(),
)

In [ ]:
# Upload an image or use a test image
# Option 1: Upload from your computer
from google.colab import files
uploaded = files.upload()  # Click to upload a leaf image
image_path = list(uploaded.keys())[0]

# Option 2: Use a file from Drive
# image_path = "/content/drive/MyDrive/PhytoVeda/test_images/neem_leaf.jpg"

image = Image.open(image_path)
image

In [ ]:
# Run inference
result = pipeline.predict(image, top_k=5, generate_report=False)

print(f"Species:     {result.species_name} ({result.species_confidence:.1%})")
print(f"Pathology:   {result.pathology_label} ({result.pathology_confidence:.1%})")
print(f"Dosha:       {result.dosha.dosha.value}")
print(f"Treatments:  {result.dosha.treatments}")
print(f"Uncertain:   {result.uncertainty.is_uncertain}")

print("\nTop-5 Species:")
for name, conf in result.top_k_species:
    print(f"  {name:30s} {conf:.2%}")

print("\nTop-K Pathology:")
for name, conf in result.top_k_pathology:
    print(f"  {name:30s} {conf:.2%}")

In [ ]:
# Save prediction result to Drive
import json

result_dict = result.to_dict()
result_path = dm.results_dir / f"prediction_{Path(image_path).stem}.json"
with open(result_path, "w") as f:
    json.dump(result_dict, f, indent=2, ensure_ascii=False)
print(f"Result saved to {result_path}")

---
## 7. RAG Pipeline — Ayurvedic Text Retrieval + LLM Reports

Index classical Ayurvedic texts into ChromaDB (stored on Drive), then generate comprehensive botanical reports.

### 7a. Upload Ayurvedic Texts

Place your text files in the Drive texts directory. Expected structure:
```
Drive/PhytoVeda/texts/
    charaka_samhita/
        chapter1.txt
        chapter2.txt
    susruta_samhita/
        ...
    vrikshayurveda/
        ...
    pharmacopoeia/
        ...
```

In [ ]:
from phytoveda.rag.indexer import AyurvedicTextIndexer

# Check if texts exist on Drive
texts_dir = dm.texts_dir
if texts_dir.exists() and any(texts_dir.rglob("*.txt")):
    txt_count = len(list(texts_dir.rglob("*.txt")))
    print(f"Found {txt_count} text files in {texts_dir}")
else:
    print(f"No texts found at {texts_dir}")
    print("Upload .txt files to your Drive under PhytoVeda/texts/ to enable RAG.")

In [ ]:
# Build ChromaDB index — stored on Drive for persistence
indexer = AyurvedicTextIndexer(texts_dir=str(dm.texts_dir))
chunks = indexer.load_and_chunk()
print(f"Total chunks: {len(chunks)}")

# Build persistent index on Drive
indexer.build_index(persist_dir=str(dm.chromadb_dir))
print(f"ChromaDB index saved to {dm.chromadb_dir}")

In [ ]:
# Test retrieval
from phytoveda.rag.retriever import AyurvedicRetriever

retriever = AyurvedicRetriever(persist_dir=str(dm.chromadb_dir))
results = retriever.retrieve_for_diagnosis(
    species_name="Azadirachta indica",
    pathology_label="Bacterial Spot",
    top_k=3,
)

for i, r in enumerate(results, 1):
    print(f"\n--- Source {i}: {r.source} / {r.chapter} (score: {r.score:.3f}) ---")
    print(r.text[:200] + "...")

### 7b. Generate LLM Report

Choose your LLM provider. Set the API key below.

In [ ]:
# ── Choose ONE provider ──

# Option 1: Gemini (default)
LLM_PROVIDER = "gemini"
LLM_API_KEY = ""  # @param {type:"string"}

# Option 2: Claude
# LLM_PROVIDER = "claude"
# LLM_API_KEY = "sk-ant-api03-..."  # @param {type:"string"}

# Option 3: OpenAI
# LLM_PROVIDER = "openai"
# LLM_API_KEY = "sk-proj-..."  # @param {type:"string"}

# Option 4: Llama (via Together.ai or local Ollama — not typical for Colab)
# LLM_PROVIDER = "llama"
# LLM_API_KEY = "tog-..."  # @param {type:"string"}
# LLM_BASE_URL = "https://api.together.xyz/v1"  # for Together.ai

In [ ]:
from phytoveda.rag.report_generator import ReportGenerator
from phytoveda.vrikshayurveda.mapper import VrikshayurvedaMapper

# Get Dosha assessment from the inference result
mapper = VrikshayurvedaMapper()
dosha = mapper.assess(result.pathology_label, result.pathology_confidence)

print(f"Species:   {result.species_name}")
print(f"Pathology: {result.pathology_label}")
print(f"Dosha:     {dosha.dosha.value} (confidence: {dosha.confidence:.1%})")

In [ ]:
# Generate comprehensive report
generator = ReportGenerator(provider=LLM_PROVIDER, api_key=LLM_API_KEY)

if LLM_API_KEY:
    # Full LLM report with RAG context
    report = generator.generate(
        species_name=result.species_name,
        pathology_label=result.pathology_label,
        dosha=dosha,
        retrieved_contexts=results,  # from retriever above
        confidence=result.species_confidence,
    )
else:
    # Offline report (no API key needed)
    print("No API key — generating offline report")
    report = generator.generate_offline(
        species_name=result.species_name,
        pathology_label=result.pathology_label,
        dosha=dosha,
    )

# Display as Markdown
from IPython.display import Markdown, display
display(Markdown(report.to_markdown()))

In [ ]:
# Save report to Drive (both JSON and Markdown)
species_slug = result.species_name.lower().replace(" ", "_")

json_path = dm.save_report(report.to_json(), f"{species_slug}_report.json")
md_path = dm.save_report(report.to_markdown(), f"{species_slug}_report.md")

print(f"JSON report: {json_path}")
print(f"Markdown report: {md_path}")

---
## 8. Active Learning Pipeline

Process uncertain predictions through the LLM oracle and human expert queue. All quarantine data persists to Drive.

In [ ]:
from phytoveda.active_learning.quarantine import QuarantineManager
from phytoveda.active_learning.oracle import LLMOracle, HumanExpertQueue, OraclePipeline

# Quarantine on Drive (persistent)
quarantine = QuarantineManager(quarantine_dir=str(dm.quarantine_dir))
print(f"Quarantine directory: {dm.quarantine_dir}")
print(f"Pending:  {quarantine.pending_count}")
print(f"Labeled:  {quarantine.labeled_count}")
print(f"Total:    {quarantine.total_count}")

In [ ]:
# Set up oracle pipeline
llm_oracle = None
if LLM_API_KEY:
    llm_oracle = LLMOracle(provider=LLM_PROVIDER, api_key=LLM_API_KEY)
    print(f"LLM Oracle: {LLM_PROVIDER}")
else:
    print("No API key — LLM oracle disabled, routing all to human queue")

expert_queue = HumanExpertQueue(queue_dir=str(dm.expert_queue_dir))
print(f"Expert queue: {dm.expert_queue_dir}")
print(f"  Pending: {expert_queue.pending_count}")
print(f"  Labeled: {expert_queue.labeled_count}")

oracle_pipeline = OraclePipeline(
    quarantine=quarantine,
    llm_oracle=llm_oracle,
    expert_queue=expert_queue,
    llm_confidence_threshold=0.7,
)

In [ ]:
# Process pending quarantine entries
if quarantine.pending_count > 0:
    stats = oracle_pipeline.process_pending()
    print(f"Oracle results:")
    print(f"  LLM labeled:   {stats['llm_labeled']}")
    print(f"  Human routed:  {stats['human_routed']}")
    print(f"  Errors:        {stats['errors']}")
else:
    print("No pending quarantine entries to process.")

In [ ]:
# Export labeled data for retraining
retraining_data = quarantine.export_for_retraining()
print(f"Retraining samples available: {len(retraining_data)}")

if retraining_data:
    for path, species, pathology in retraining_data[:5]:
        print(f"  {Path(path).name}: {species} / {pathology}")

---
## 9. Enterprise Features

### 9a. Conservation Monitoring (IUCN Red List)

In [ ]:
from phytoveda.conservation.iucn import ConservationRegistry

registry = ConservationRegistry()

# Check identified species
alert = registry.check(result.species_name)
if alert:
    print(f"CONSERVATION ALERT")
    print(f"  Species:  {result.species_name}")
    print(f"  Status:   {alert.status.name}")
    print(f"  Severity: {alert.severity}")
    print(f"  Message:  {alert.message}")
    print(f"  Harvest:  {'Allowed' if alert.harvest_allowed else 'PROHIBITED'}")
else:
    print(f"{result.species_name}: No conservation concern (or not in registry)")

# List all threatened species
print("\nThreatened species in registry:")
for sp in registry.get_threatened_species():
    print(f"  {sp.status.name:3s}: {sp.scientific_name} — {sp.notes}")

### 9b. Supply Chain Traceability

In [ ]:
from phytoveda.traceability.events import EventLedger

# Ledger on Drive for tamper-proof persistence
ledger = EventLedger(persist_path=str(dm.ledger_path))

# Log the identification event
event = ledger.record(
    species=result.species_name,
    pathology=result.pathology_label,
    species_confidence=result.species_confidence,
    dosha=result.dosha.dosha.value,
    # GPS coordinates (example — replace with actual location)
    latitude=12.9716,
    longitude=77.5946,
    operator_id="colab-user",
    batch_id="BATCH-2026-001",
)

print(f"Event logged: {event.event_hash[:16]}...")
print(f"Chain valid:  {ledger.verify_chain()}")
print(f"Total events: {ledger.event_count}")
print(f"Ledger path:  {dm.ledger_path}")

In [ ]:
# Biodiversity summary
summary = ledger.biodiversity_summary()
print("Biodiversity Summary:")
for species, count in summary.items():
    print(f"  {species}: {count} identifications")

### 9c. Formulation Validation

In [ ]:
from phytoveda.formulation.validator import FormulationValidator, IdentifiedHerb

validator = FormulationValidator()

# Example: Validate a Triphala formulation
herbs = [
    IdentifiedHerb("Terminalia chebula", "Healthy", 0.95, is_healthy=True),
    IdentifiedHerb("Terminalia bellirica", "Healthy", 0.91, is_healthy=True),
    IdentifiedHerb("Emblica officinalis", "Powdery Mildew", 0.88, is_healthy=False),
]

result_formulation = validator.validate("Triphala", herbs)
print(f"Formulation: {result_formulation.formulation.name}")
print(f"Quality:     {result_formulation.overall_quality}")
print(f"\nHerb Status:")
for hr in result_formulation.herb_results:
    print(f"  {hr.required_herb.scientific_name:30s} {hr.status}")
if result_formulation.warnings:
    print(f"\nWarnings:")
    for w in result_formulation.warnings:
        print(f"  {w}")

---
## 10. Storage Summary

Check what's saved on Drive and SSD.

In [ ]:
# Disk usage report
print(dm.summary())
print("\nDisk Usage:")
for name, size in dm.disk_usage().items():
    print(f"  {name:30s} {size}")

In [ ]:
# List saved checkpoints
print("Saved checkpoints:")
for ckpt in dm.list_checkpoints():
    size_mb = ckpt.stat().st_size / (1024 ** 2)
    print(f"  {ckpt.name:30s} {size_mb:.1f} MB")

In [ ]:
# List saved reports
print("Saved reports:")
for report_file in sorted(dm.reports_dir.glob("*")):
    print(f"  {report_file.name}")

---
## 11. Launch API Server (Optional)

Start the FastAPI inference server directly from Colab. Uses `ngrok` for external access.

In [ ]:
# Install ngrok for tunneling
!pip install -q pyngrok

from pyngrok import ngrok
import subprocess
import threading

# Start server in background
best_ckpt = str(dm.latest_checkpoint())
cmd = [
    "python", "-m", "phytoveda.api.server",
    "--checkpoint", best_ckpt,
    "--host", "0.0.0.0",
    "--port", "8000",
]

# Add RAG + LLM if available
if dm.chromadb_dir.exists() and any(dm.chromadb_dir.iterdir()):
    cmd.extend(["--chromadb-dir", str(dm.chromadb_dir)])
if LLM_API_KEY:
    cmd.extend(["--gemini-api-key", LLM_API_KEY])

server = subprocess.Popen(cmd)

# Create public URL
import time
time.sleep(3)  # Wait for server startup
public_url = ngrok.connect(8000)
print(f"\nPhytoVeda API is live at: {public_url}")
print(f"  Health check: {public_url}/health")
print(f"  Identify:     curl -X POST {public_url}/identify -F 'file=@leaf.jpg'")

In [ ]:
# Stop the server when done
# server.terminate()
# ngrok.disconnect(public_url)

---

## Quick Reference

### Storage Layout
| Location | Type | Contents |
|----------|------|----------|
| `/content/datasets/` | SSD (fast) | Downloaded datasets |
| `/content/cache/` | SSD (fast) | Processing temp files |
| `Drive/PhytoVeda/checkpoints/` | Drive (persistent) | Model weights |
| `Drive/PhytoVeda/results/` | Drive (persistent) | Evaluation metrics, plots |
| `Drive/PhytoVeda/chromadb/` | Drive (persistent) | RAG vector store |
| `Drive/PhytoVeda/reports/` | Drive (persistent) | Botanical reports |
| `Drive/PhytoVeda/quarantine/` | Drive (persistent) | Active learning images |
| `Drive/PhytoVeda/traceability/` | Drive (persistent) | Supply chain events |
| `Drive/PhytoVeda/texts/` | Drive (persistent) | Ayurvedic source texts |
| `Drive/PhytoVeda/configs/` | Drive (persistent) | Saved training configs |

### Runtime Restart Recovery
After a Colab runtime restart, re-run cells 1a through 1d to remount Drive and reinitialize. All persistent data (checkpoints, results, ChromaDB, reports) will still be on Drive. Only datasets need re-downloading to SSD.